# Load Films data after Extraction

In [3]:
import pandas as pd


# Load the datasets
file_path = r"C:\Users\Ezra\Downloads\film_web_imputation_26_01_2025_5K_Samples.csv"


df = pd.read_csv(file_path)

# Display the column names and a sample of the data
df_info = df.head()
df_columns = df.columns.tolist()

(df_info, df_columns)

(                                     film   filmLabel  boxOffice  \
 0       http://dbpedia.org/resource/$pent       $pent     9287.0   
 1       http://dbpedia.org/resource/$pent       $pent     9287.0   
 2       http://dbpedia.org/resource/$pent       $pent     9287.0   
 3  http://dbpedia.org/resource/'71_(film)  '71 (film)  3100000.0   
 4  http://dbpedia.org/resource/'71_(film)  '71 (film)  3100000.0   
 
   boxOfficeClass  directorLabel              castLabel     budget releaseDate  \
 0             No  Gil Cates Jr.           Jason London        NaN         NaN   
 1             No  Gil Cates Jr.      Charlie Spradling        NaN         NaN   
 2             No  Gil Cates Jr.            Phill Lewis        NaN         NaN   
 3             No   Yann Demange       Sam Reid (actor)  8100000.0         NaN   
 4             No   Yann Demange  Paul Anderson (actor)  8100000.0         NaN   
 
         genre  director's previous success rate  ...  \
 0       Drama                   

In [4]:
# Calculate missing values per column
missing_counts = df.isnull().sum()
missing_percentages = (missing_counts / len(df)) * 100

# Combine results into a DataFrame for readability
missing_summary = pd.DataFrame({
    'Missing Values': missing_counts,
    'Percentage (%)': missing_percentages.round(2)
}).sort_values(by='Percentage (%)', ascending=False)

# Print the summary
print("Missing values per column:")
print(missing_summary)


Missing values per column:
                                           Missing Values  Percentage (%)
releaseYear                                          4761          100.00
hasSequel                                            4761          100.00
productionCompanyCount                               4761          100.00
languageCount                                        4761          100.00
runtime                                              4761          100.00
movieLabel                                           4761          100.00
movie                                                4761          100.00
releaseDate                                          4280           89.90
budget                                               1696           35.62
directorLabel                                         541           11.36
castLabel                                             124            2.60
film festival awards won                                0            0.00
film       

In [85]:
df['boxOfficeClass'].value_counts()

No     4263
Yes     498
Name: boxOfficeClass, dtype: int64

# Initial Data modeling

In [5]:
import pandas as pd


# Load the datasets
file_path = r"C:\Users\Ezra\Downloads\film_web_imputation_26_01_2025_5K_Samples.csv"



df = pd.read_csv(file_path)

# Display the column names and a sample of the data
df_info = df.head()
df_columns = df.columns.tolist()

(df_info, df_columns)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import xgboost as xgb


columns_to_drop = [ #columns that has benn achieved by Extraction
    "director's previous success rate",
    "cast's previous success rate",
    'average audience rating for similar films',
    'social media buzz score',
    'film festival awards won'
]

df = df.drop(columns=columns_to_drop, axis=1, errors='raise')  # 'raise' is default, you can also use 'ignore'
print(df.columns.tolist())



# Drop all-empty or irrelevant columns except filmLabel
df.drop(columns=[
    'film', 'movie', 'movieLabel', 'runtime',
    'languageCount', 'productionCompanyCount', 'hasSequel', 'releaseYear'
], inplace=True)

# Drop duplicates (same movie with different cast members)
#df = df.drop_duplicates(subset=['filmLabel'])

# Now safely drop filmLabel
df.drop(columns=['filmLabel'], inplace=True)

# Convert target variable to binary
df['boxOfficeClass'] = df['boxOfficeClass'].map({'Yes': 1, 'No': 0})

# Fill missing values
df['budget'].fillna(df['budget'].median(), inplace=True)
df['releaseDate'] = pd.to_datetime(df['releaseDate'], errors='coerce')
df['releaseYear'] = df['releaseDate'].dt.year.fillna(2000).astype(int)
df['directorLabel'].fillna('Unknown', inplace=True)
df['castLabel'].fillna('Unknown', inplace=True)

# Smart feature engineering
# One-hot encode top 5 genres
df['genre'] = df['genre'].fillna('Unknown')
top_genres = df['genre'].value_counts().nlargest(5).index
df['genre'] = df['genre'].apply(lambda g: g if g in top_genres else 'Other')
df = pd.get_dummies(df, columns=['genre'], drop_first=True)

# Drop string columns
df.drop(columns=['castLabel', 'directorLabel', 'releaseDate'], inplace=True)

# Fill any remaining NaNs
df.fillna(0, inplace=True)

# Prepare features and target
X = df.drop(columns=['boxOfficeClass', 'boxOffice'])
y = df['boxOfficeClass']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# Handle class imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
model = xgb.XGBClassifier(
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
f1 = f1_score(y_test, y_pred)
print(f"F1 Score: {f1:.4f}")


['film', 'filmLabel', 'boxOffice', 'boxOfficeClass', 'directorLabel', 'castLabel', 'budget', 'releaseDate', 'genre', 'movie', 'movieLabel', 'runtime', 'languageCount', 'productionCompanyCount', 'hasSequel', 'releaseYear']
F1 Score: 0.6481


# Imputated Data Modeling - Without Prioritization

In [17]:
import pandas as pd


# Load the datasets
file_path = r"C:\Users\Ezra\Downloads\film_web_imputation_26_01_2025_5K_Samples.csv"



df = pd.read_csv(file_path)

# Display the column names and a sample of the data
df_info = df.head()
df_columns = df.columns.tolist()


import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import xgboost as xgb



# Drop all-empty or irrelevant columns except filmLabel
df.drop(columns=[
    'film', 'movie', 'movieLabel', 'runtime',
    'languageCount', 'productionCompanyCount', 'hasSequel', 'releaseYear'
], inplace=True)

# Drop duplicates (same movie with different cast members)
#df = df.drop_duplicates(subset=['filmLabel'])

# Now safely drop filmLabel
df.drop(columns=['filmLabel'], inplace=True)

# Convert target variable to binary
df['boxOfficeClass'] = df['boxOfficeClass'].map({'Yes': 1, 'No': 0})

# Fill missing values
df['budget'].fillna(df['budget'].median(), inplace=True)
df['releaseDate'] = pd.to_datetime(df['releaseDate'], errors='coerce')
df['releaseYear'] = df['releaseDate'].dt.year.fillna(2000).astype(int)
df['directorLabel'].fillna('Unknown', inplace=True)
df['castLabel'].fillna('Unknown', inplace=True)

# Smart feature engineering
df['budget_to_buzz_ratio'] = df['budget'] / (df['social media buzz score'] + 1)
df['buzz_x_awards'] = df['social media buzz score'] * df['film festival awards won']
df['cast_x_director_score'] = df["cast's previous success rate"] * df["director's previous success rate"]

# One-hot encode known vs unknown
df['known_director'] = (df['directorLabel'] != 'Unknown').astype(int)
df['known_cast'] = (df['castLabel'] != 'Unknown').astype(int)

# One-hot encode top 5 genres
df['genre'] = df['genre'].fillna('Unknown')
top_genres = df['genre'].value_counts().nlargest(5).index
df['genre'] = df['genre'].apply(lambda g: g if g in top_genres else 'Other')
df = pd.get_dummies(df, columns=['genre'], drop_first=True)

# Drop string columns
df.drop(columns=['castLabel', 'directorLabel', 'releaseDate'], inplace=True)

# Fill any remaining NaNs
df.fillna(0, inplace=True)

# Prepare features and target
X = df.drop(columns=['boxOfficeClass', 'boxOffice'])
y = df['boxOfficeClass']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# Handle class imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
model = xgb.XGBClassifier(
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
f1 = f1_score(y_test, y_pred)
print(f"F1 Score: {f1:.4f}")


F1 Score: 0.8384


In [18]:
df.drop('boxOffice', axis=1, inplace=True)

## Imputed Data Modeling - With Prioritization

In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from src.dataset_with_indices_for_full_and_partial_data import Index_Dataset
from src.filtering_methods.filtering_methods_return_index import (RandomSampleFilter, ActiveLearningStartFromCoreSet, FilterEachIterXgboostPathSampleFinal)
import xgboost as xgb

random_state = 42


# ---------- 1. Load & sanitize once ----------
target_col = "boxOfficeClass"
df_original = df


feature_cols        = [c for c in df_original.columns if c != target_col]
#df_original.columns = (   sanitize_column_names(feature_cols) + [target_col])  # keep label untouched

# ---------- 2. Build X / y ----------
X = df_original.drop(columns=[target_col]).fillna(-999).astype(np.float32)
y = df_original[target_col]

# ---------- 3. Train / test split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=random_state
)

# ---------- 4. Feed sanitized structures to Index_Dataset ----------
dataset_partial = Index_Dataset(
    df_original,      # contains the SAME sanitized names
    X_train,
    y_train,
    X_test,
    y_test,
    target_col,
    f1_score,
)

# ---------- 5. Run your filter ----------
all_results   = []
n = len(X_train)
print(n)
core_set_sizes = [n//10, n//5, n // 3, n // 2, int(0.75 * n), n]

for size in core_set_sizes:
    random_state += 1
    p          = size / n

    # Handle class imbalance
    #scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=42
    )

    sampler    = RandomSampleFilter(name="RandomSampleFilter",
                                    p=p, model=model)
    sampler.test_indices_filter_method(
        dataset_partial, dataset_partial, print_results=True
    )
    all_results.append(
        {"Method": "RandomSampleFilter",
         "Core Set Size": size,
         "F1 Score": sampler.scores,
         }
    )

    initial_set_size = size //2
    # ExtendTAB Filter(Active Learning Start From Core Set)
    core_tab = FilterEachIterXgboostPathSampleFinal(
        'test_each_iter',
        trees_to_stop=30,
        threshold=5,
        examples_to_keep=size//2,
        prediction_model=model
    )
    core_set_sampler = core_tab.sample_indices(
        dataset_partial,
        p = initial_set_size / len(dataset_partial.X_train)
    )

    sampler = ActiveLearningStartFromCoreSet(name="ExtenTab",number_of_examples=size, start_size=size//2,core_set_sampler=core_set_sampler, batch_size=100,
                                     model=model)
    sampler.test_indices_filter_method(
        dataset_partial, dataset_partial, print_results=True
    )
    all_results.append(
        {"Method": "ExtenTab",
         "Core Set Size": size,
         "F1 Score": sampler.scores,}
    )


3570
0 0.1
Random Sampler: 0 268
FilteringResults(modeling_atitude='sample by partial dataset, train and test by the full dataset', score=0.5361702127659574, filtered_score=0.5361702127659574, run_time=0, new_size_percent=0.07507002801120448)
ExtendTab: len(initial_indices) 178 start_size 178
start size 178
ExtendTab: AL Phase Current training set size: 178
ExtendTab: AL Phase Current training set size: 278
validate that the training set size is the wished size training set size: 378 wished core-set size: 357
FilteringResults(modeling_atitude='sample by partial dataset, train and test by the full dataset', score=0.5523255813953487, filtered_score=0.5523255813953487, run_time=0, new_size_percent=0.10588235294117647)
0 0.2
Random Sampler: 0 479
FilteringResults(modeling_atitude='sample by partial dataset, train and test by the full dataset', score=0.6301369863013699, filtered_score=0.6301369863013699, run_time=0, new_size_percent=0.1341736694677871)
ExtendTab: len(initial_indices) 357 st

In [20]:
# Step 1: Flatten F1 scores into rows
flattened = []
for entry in all_results:
    method = entry['Method']
    core_size = entry['Core Set Size']
    for score in entry['F1 Score']:
        flattened.append({'Method': method, 'Core Set Size': core_size, 'F1 Score': score})

df = pd.DataFrame(flattened)
# Step 2: Group and summarize
summary = df.groupby(['Method', 'Core Set Size']).agg(
    avg_f1_score=('F1 Score', 'mean'),
).reset_index()


# Step 3: Print or export
(summary.sort_values(['Method', 'Core Set Size', 'avg_f1_score'], ascending=[True, True,False]))

,Method,Core Set Size,avg_f1_score
0,ExtenTab,357,0.552326
1,ExtenTab,714,0.426850
2,ExtenTab,1190,0.560920
3,ExtenTab,1785,0.664850
4,ExtenTab,2677,0.672176
5,ExtenTab,3570,0.664850
6,RandomSampleFilter,357,0.536170
7,RandomSampleFilter,714,0.630137
8,RandomSampleFilter,1190,0.689076
9,RandomSampleFilter,1785,0.770492


|

## ExtendTab Free Size

In [14]:
from src.filtering_methods.filtering_methods_return_index import (ActiveLearningStartFromCoreSet_based_on_reccommended_sampling_sizes, FilterEachIterXgboostPathSampleFinal)


# ---------- 1. Load & sanitize once ----------
target_col = "boxOfficeClass"
df_original = df


feature_cols        = [c for c in df_original.columns if c != target_col]
#df_original.columns = (   sanitize_column_names(feature_cols) + [target_col])  # keep label untouched

# ---------- 2. Build X / y ----------
X = df_original.drop(columns=[target_col]).fillna(-999).astype(np.float32)
y = df_original[target_col]

# ---------- 3. Train / test split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=random_state
)

# ---------- 4. Feed sanitized structures to Index_Dataset ----------
dataset_partial = Index_Dataset(
    df_original,      # contains the SAME sanitized names
    X_train,
    y_train,
    X_test,
    y_test,
    target_col,
    f1_score)

# ===============================
#   ExtendTAB (Free size)
# ===============================
core_tab_hard_Samples = FilterEachIterXgboostPathSampleFinal(
    'test_each_iter',
    trees_to_stop=30,
    threshold=5,
    prediction_model=model,
    examples_to_keep=len(dataset_partial.X_train)//2,
)
core_set_sampler = core_tab_hard_Samples.sample_indices(
    dataset_partial,
    p=initial_set_size / len(dataset_partial.X_train)
)

filter_instance = ActiveLearningStartFromCoreSet_based_on_reccommended_sampling_sizes(
    name='ExtendTAB',
    start_size=len(dataset_partial.X_train)//2,
    number_of_examples=len(dataset_partial.X_train),
    model=model,
    core_set_sampler=core_set_sampler,
    batch_size=100
)

# Again, the critical fix: call the method
filter_instance.test_indices_filter_method(dataset_partial, dataset_partial, print_results=True)

all_results.append({
    #'Dataset': dataset_name,
    'Method': 'Extentab Free size',
    'Core Set Size': filter_instance.number_of_examples,
    'F1 Score': filter_instance.scores,
    'Trial': filter_instance.trials_number,
     'Data Used (%)': filter_instance.new_size_percents * 100
})


[ExtendTab] using supplied core set of 1785/1785
[ExtendTab] AL-phase — train size 1785
[ExtendTab] val-score 0.9669
[ExtendTab] AL-phase — train size 1885
[ExtendTab] val-score 0.9669
[ExtendTab] AL-phase — train size 1985
[ExtendTab] val-score 0.9669
[ExtendTab] AL-phase — train size 2085
[ExtendTab] val-score 0.9712
[ExtendTab] AL-phase — train size 2185
[ExtendTab] val-score 1.0000
[ExtendTab] AL-phase — train size 2285
[ExtendTab] val-score 1.0000
[ExtendTab] AL-phase — train size 2385
[ExtendTab] val-score 1.0000
[ExtendTab] AL-phase — train size 2485
[ExtendTab] val-score 1.0000
[ExtendTab] AL-phase — train size 2585
[ExtendTab] val-score 1.0000
[ExtendTab] AL-phase — train size 2685
[ExtendTab] val-score 1.0000
[ExtendTab] Stopped after 5 flat rounds.
[ExtendTab] initial 1785 → final 2785 (time 5.7s)
FilteringResults(modeling_atitude='sample by partial dataset, train and test by the full dataset', score=1.0, filtered_score=1.0, run_time=0, new_size_percent=0.7801120448179272)
